# Can we predict `passed_quiz`?
Target: whether a student passes the quiz closing a tutoring session (one row = one session).

Plan: inspect the data, then evaluate models with **student-grouped** cross-validation. The same student appears in ~7 sessions on average, so a random row split would let the model see a student's other sessions during training and overstate performance for new students. We report ROC AUC (threshold-free), accuracy against the majority-class baseline, and log loss.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False)); df.describe().T

## 1. Data checks

In [ ]:
g=df.groupby('student_key')
print(df.shape, 'missing:',int(df.isna().sum().sum()), 'duplicate rows:',int(df.duplicated().sum()))
print('students:',df.student_key.nunique(), '| sessions per student (min/median/max):', g.size().agg(['min','median','max']).tolist())
print('pass rate:',df.passed_quiz.mean().round(3))
print('prior_score > 100:',(df.prior_score>100).sum(),'rows (max',df.prior_score.max(),') - kept; not clearly an error')
print('enrol_year / school_code constant within student:',(g.enrol_year.nunique()==1).all(),(g.school_code.nunique()==1).all())
print('prior_score varies within student: mean range',(g.prior_score.max()-g.prior_score.min()).mean().round(2))

`prior_score` is stored on the *session record* and varies a little across one student's sessions, so it is a noisy copy of a student-level placement score. `enrol_year` and `school_code` are student-level.

## 2. Univariate relationships

In [ ]:
fig,ax=plt.subplots(1,4,figsize=(17,3.5))
df.groupby('hints_used').passed_quiz.mean().plot(marker='o',ax=ax[0],title='pass rate by hints_used')
df.groupby(pd.qcut(df.session_minutes,10)).passed_quiz.mean().reset_index(drop=True).plot(marker='o',ax=ax[1],title='by session_minutes decile')
df.groupby(pd.qcut(df.prior_score,10)).passed_quiz.mean().reset_index(drop=True).plot(marker='o',ax=ax[2],title='by prior_score decile')
df.groupby('enrol_year').passed_quiz.mean().plot(marker='o',ax=ax[3],title='by enrol_year')
for a in ax: a.set_ylim(0.2,0.65)
plt.tight_layout(); plt.show()
sp=g.passed_quiz.mean(); n=g.size(); pr=df.passed_quiz.mean()
print('sd of student-level pass rates: %.2f; expected from chance alone given session counts: %.2f'%(sp.std(), np.sqrt((pr*(1-pr)/n).mean())))

More hints go with lower pass rates; longer sessions with higher. `prior_score` shows almost no marginal relationship. Student-level pass rates are spread far more than chance, so there is a persistent student effect.

## 3. Models for a new, unseen student (5-fold, grouped by student)

In [ ]:
y=df.passed_quiz.values; groups=df.student_key.values
feats=['prior_score','session_minutes','hints_used','enrol_year','school_code']
X=df[feats]
cv=GroupKFold(n_splits=5)
res=[];P={}
for name,m,Xs in [
 ('majority class',DummyClassifier(strategy='prior'),X),
 ('logistic (minutes, hints)',make_pipeline(StandardScaler(),LogisticRegression()),df[['session_minutes','hints_used']]),
 ('logistic (all 5 features)',make_pipeline(StandardScaler(),LogisticRegression()),X),
 ('logistic + school one-hot',make_pipeline(ColumnTransformer([('n',StandardScaler(),['prior_score','session_minutes','hints_used','enrol_year']),('s',OneHotEncoder(handle_unknown='ignore'),['school_code'])]),LogisticRegression(C=0.3,max_iter=1000)),X),
 ('gradient boosting (all 5)',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,categorical_features=[4]),X)]:
    p=cross_val_predict(m,Xs,y,cv=cv,groups=groups,method='predict_proba')[:,1]; P[name]=p
    res.append(dict(model=name,auc=roc_auc_score(y,p),accuracy=accuracy_score(y,p>0.5),logloss=log_loss(y,p)))
pd.DataFrame(res).round(3)

In [ ]:
lr=make_pipeline(StandardScaler(),LogisticRegression()).fit(X,y)
print(pd.Series(lr[-1].coef_[0],index=feats).round(3).to_string(),'\n(standardised logistic coefficients, full data, direction only)')
s=df.groupby('school_code').passed_quiz.agg(['mean','size']); print('\nschool pass-rate range:',s['mean'].min().round(2),'-',s['mean'].max().round(2),'; sessions per school',s['size'].min(),'-',s['size'].max())

## 4. Returning students: does a student's history help?
Section 3 answers "a student we have never seen". If we predict for students already in the data, their earlier sessions are available. We use random 5-fold by session and add a smoothed pass rate over the student's *training* sessions only (leave-one-out for training rows, so a row's own label never feeds its own feature).

In [ ]:
def add_hist(tr,te,prior=df.passed_quiz.mean(),k=2):
    st=tr.groupby('student_key').passed_quiz.agg(['sum','count'])
    a=tr.join(st,on='student_key'); loo=(a['sum']-a.passed_quiz+k*prior)/(a['count']-1+k)
    b=te.join(st,on='student_key'); h=(b['sum'].fillna(0)+k*prior)/(b['count'].fillna(0)+k)
    return loo.values,h.values
cols=['prior_score','session_minutes','hints_used']
out={'no history':np.zeros(len(df)),'with student history':np.zeros(len(df))}
for tr_i,te_i in KFold(5,shuffle=True,random_state=0).split(df):
    tr,te=df.iloc[tr_i],df.iloc[te_i]
    m=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200)
    out['no history'][te_i]=m.fit(tr[cols],tr.passed_quiz).predict_proba(te[cols])[:,1]
    htr,hte=add_hist(tr,te)
    out['with student history'][te_i]=m.fit(tr[cols].assign(h=htr),tr.passed_quiz).predict_proba(te[cols].assign(h=hte))[:,1]
pd.DataFrame([dict(setup=k,auc=roc_auc_score(y,p),accuracy=accuracy_score(y,p>.5),logloss=log_loss(y,p)) for k,p in out.items()]).round(3)

## 5. Calibration and fold stability (new-student gradient boosting)

In [ ]:
p=P['gradient boosting (all 5)']
b=pd.qcut(p,10,duplicates='drop'); c=pd.DataFrame({'pred':p,'obs':y}).groupby(b,observed=True).mean()
plt.figure(figsize=(4.5,4)); plt.plot(c.pred,c.obs,'o-'); lo,hi=min(c.pred.min(),c.obs.min()),max(c.pred.max(),c.obs.max()); plt.plot([lo,hi],[lo,hi],'k--')
plt.xlabel('mean predicted'); plt.ylabel('observed pass rate'); plt.title('calibration (decile bins)'); plt.show()
aucs=[roc_auc_score(y[i],p[i]) for _,i in cv.split(X,y,groups)]; print('AUC per fold:',np.round(aucs,3))

## Conclusions (numbers from running these cells; AUC ranges are across the 5 folds)
- **New students: only weakly predictable.** Grouped-CV AUC is about 0.60 (folds 0.56-0.59 for boosting) and accuracy about 0.60 against a 0.58 majority baseline. Signal comes from `session_minutes` (+) and `hints_used` (-). `prior_score`, `enrol_year` and `school_code` add essentially nothing; the more flexible models (school one-hot, boosting) do worse, i.e. they overfit. Not good enough to act on for individual decisions.
- **Returning students: much more predictable (AUC ~0.84, accuracy ~0.77)** once the student's own past pass rate is used. Most of the predictable variation is a persistent per-student effect that none of the recorded attributes explains. Caveat: this split is by session, so it only applies when the student already has prior sessions at prediction time. Within-student ordering by time is not available (no session date), so past/future cannot be enforced; if sessions are not exchangeable over time, the figure may be optimistic.
- Not tested: interactions with time, other quiz context, or anything absent from the dataset. 19 `prior_score` values exceed 100; they were kept.